# Foundation Models & Scaling Laws

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/transformers/05-foundation-models-and-scaling

A runnable tour of the Chinchilla loss law $L(N, D) = A/N^\alpha + B/D^\beta + E$:
we fit it from synthetic (N, D, L) measurements, derive the compute-optimal
allocation $D \approx 20 N$ from the fit, and plot loss-vs-compute curves on a
log-log axis with the compute-optimal points marked. Pure NumPy + matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d27'
plt.rcParams['text.color']       = '#e2e8f0'
plt.rcParams['axes.labelcolor']  = '#e2e8f0'
plt.rcParams['xtick.color']      = '#94a3b8'
plt.rcParams['ytick.color']      = '#94a3b8'
plt.rcParams['axes.edgecolor']   = '#334155'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.color']       = '#1e293b'
plt.rcParams['figure.figsize']   = (9, 5)
np.random.seed(0)

## Intuition — loss is predictable before you train

The strangest empirical fact about large models: their loss follows a smooth **power law** in
parameters `N` and training tokens `D` — `L(N, D) = A/N^α + B/D^β + E` (the **Chinchilla** form),
where `E` is the irreducible floor. This makes trillion-dollar questions answerable with arithmetic:
fit the law from a *sweep of small runs*, then extrapolate to predict a big model's loss and derive the
**compute-optimal split** — for a FLOPs budget `C ≈ 6ND`, how big a model, trained on how many tokens?
Chinchilla's answer (~20 tokens per parameter) reshaped the field. We fit the law from synthetic
measurements, derive the optimum, and verify the closed form against brute-force search.

## 1 — The Chinchilla loss law

The Hoffmann et al. (2022) joint fit:

$$L(N, D) = \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}} + E$$

with $N$ = parameter count, $D$ = training tokens, and rough exponents
$\alpha \approx 0.34$, $\beta \approx 0.28$. We start by evaluating it on a
grid and plotting some curves to build intuition.

In [ ]:
# Ground-truth constants we will pretend are the unknowns we are fitting.
A_true, B_true, E_true = 406.4, 410.7, 1.69
alpha_true, beta_true  = 0.34, 0.28

def chinchilla_loss(N, D, A=A_true, B=B_true, E=E_true,
                    alpha=alpha_true, beta=beta_true):
    return A / N**alpha + B / D**beta + E

# Quick sanity check at a handful of (N, D) combos.
for N, D in [(1e9, 2e10), (7e9, 1.4e11), (7e10, 1.4e12)]:
    print(f"N={N:.0e}, D={D:.0e}  ->  L = {chinchilla_loss(N, D):.3f}")

**What to notice:** the loss law has three terms with clean meanings — a **model-capacity** term
`A/N^α` (shrinks with parameters), a **data** term `B/D^β` (shrinks with tokens), and the
**irreducible entropy** `E ≈ 1.69` that no scale can beat. Every (N, D) evaluation is just this
three-term sum.

## 2 — Fitting the law from synthetic measurements

Imagine we have run a sweep of (N, D) training jobs and recorded the final loss.
We will treat $A$, $B$, $E$ as unknowns at fixed exponents (a common practical
simplification) and recover them via plain NumPy least squares.

Writing $u = N^{-\alpha}$ and $v = D^{-\beta}$, the model is **linear** in
$(A, B, E)$:

$$L = A \cdot u + B \cdot v + E \cdot 1.$$

So one `np.linalg.lstsq` recovers all three.

In [ ]:
# Build a noisy training-sweep dataset.
Ns = np.array([1e8, 3e8, 1e9, 3e9, 1e10, 3e10, 1e11])
Ds = np.array([1e9, 5e9, 2e10, 1e11, 5e11, 2e12])

rows = []
for N in Ns:
    for D in Ds:
        L = chinchilla_loss(N, D) * np.exp(np.random.randn() * 0.01)  # 1% noise
        rows.append((N, D, L))
rows = np.array(rows)
N_obs, D_obs, L_obs = rows[:, 0], rows[:, 1], rows[:, 2]

# Solve  A * N^-alpha + B * D^-beta + E = L   in least-squares sense.
X = np.column_stack([
    N_obs ** (-alpha_true),
    D_obs ** (-beta_true),
    np.ones_like(L_obs),
])
coef, *_ = np.linalg.lstsq(X, L_obs, rcond=None)
A_fit, B_fit, E_fit = coef

print(f"true: A={A_true:.2f}  B={B_true:.2f}  E={E_true:.3f}")
print(f"fit : A={A_fit:.2f}  B={B_fit:.2f}  E={E_fit:.3f}")

# Sanity: fitted predictions vs observed loss
pred = X @ coef
resid_pct = 100.0 * np.abs(pred - L_obs) / L_obs
print(f"mean abs residual: {resid_pct.mean():.2f}%  max: {resid_pct.max():.2f}%")

**What to notice:** from a noisy 42-run sweep, least squares recovers `A`, `B`, `E` to within a few
percent — with residuals ~1%. This is the actual methodology: measure small, fit the law, trust the
extrapolation. The scaling law turns model planning from folklore into regression.

## 3 — Deriving the compute-optimal allocation

Total training compute (forward + backward + activations) for a dense
Transformer is well approximated by

$$C \approx 6 \cdot N \cdot D \text{ FLOPs.}$$

Under the Chinchilla rule $D = 20 N$, this becomes

$$C = 6 \cdot N \cdot 20 N = 120 N^2 \;\Rightarrow\; N^* = \sqrt{C / 120}, \quad D^* = 20 N^*.$$

We will use this closed form directly. (A full derivation would set
$\partial L / \partial N = \partial L / \partial D \cdot 20$ along the constraint
$D = 20 N$ and solve; the 20 comes from the fitted exponents.)

In [ ]:
def chinchilla_optimal_demo(C):
    """Quick demo using the closed form  N* = sqrt(C / 120),  D* = 20 N*."""
    N_opt = np.sqrt(C / 120.0)
    D_opt = 20.0 * N_opt
    return N_opt, D_opt

for C in [1e21, 1e22, 1e23, 1e24]:
    N_opt, D_opt = chinchilla_optimal_demo(C)
    L_opt = chinchilla_loss(N_opt, D_opt, A=A_fit, B=B_fit, E=E_fit)
    print(f"C = {C:.0e} FLOPs  ->  N* = {N_opt/1e9:6.2f}B params,  "
          f"D* = {D_opt/1e9:7.0f}B tokens,  predicted L = {L_opt:.2f}")

**What to notice:** the compute-optimal recipe `N* = √(C/120)`, `D* = 20·N*` — about **20 tokens per
parameter** at every budget. Doubling compute should grow the model ~1.4× and the data ~1.4×, *together*.
Pre-Chinchilla models (GPT-3: 175B params, 300B tokens ≈ 1.7 tokens/param) were dramatically
under-trained by this measure.

## The library way — derive the exact optimum and verify by brute force

The quick rule `N* = √(C/120)` is a **rule of thumb** that's exact only when `α = β`. For the fitted
law (α=0.34, β=0.28) the true optimum comes from setting the derivative of
`A/N^α + B·(6N/C)^β + E` to zero:

$$N^* = \left(\frac{\alpha A}{\beta B}\right)^{\frac{1}{\alpha+\beta}} \left(\frac{C}{6}\right)^{\frac{\beta}{\alpha+\beta}}$$

The cell computes this exact optimum, verifies it against a 20,000-point brute-force sweep, and
compares with the √ rule of thumb.

In [ ]:
C = 1e23
N_grid = np.logspace(8.5, 11.5, 20000)
D_grid = C / (6 * N_grid)                                   # spend the whole budget
L_grid = chinchilla_loss(N_grid, D_grid, A=A_fit, B=B_fit, E=E_fit)
N_numeric = N_grid[L_grid.argmin()]

# exact optimum for the fitted law (alpha != beta)
a, b = alpha_true, beta_true
N_exact = ((a * A_fit) / (b * B_fit)) ** (1 / (a + b)) * (C / 6) ** (b / (a + b))
N_rule, _ = chinchilla_optimal_demo(C)                      # the sqrt rule of thumb

print(f'brute-force argmin : N* = {N_numeric/1e9:6.2f}B params')
print(f'exact formula      : N* = {N_exact/1e9:6.2f}B params')
print(f'sqrt rule of thumb : N* = {N_rule/1e9:6.2f}B params  (assumes alpha = beta)')
assert abs(np.log10(N_numeric) - np.log10(N_exact)) < 0.02, "exact formula must match brute force"
print(f'\ntokens per parameter at this optimum: {C/(6*N_numeric)/N_numeric:.0f}')
print('exact derivation == brute-force search ✓  (the sqrt rule is off because alpha != beta here)')

**What to notice:** the exact derivative-based optimum matches the brute-force search, while the
`√(C/120)` rule of thumb overshoots ~2× — because it silently assumes `α = β`, and the fitted
exponents differ (0.34 vs 0.28). With `β < α`, data is the slower-improving resource, so the optimum
shifts toward *more tokens per parameter* than 20. Two lessons in one: verify closed forms against
brute force, and know a rule of thumb's assumptions before applying it.

## 4 — Loss vs compute (log-log) with optimal points marked

For three fixed model sizes (1B, 7B, 70B parameters) we sweep $D$ from a small
token budget up to a very large one, plot $L$ vs compute $C = 6 N D$, and mark
the Chinchilla-optimal point per curve.

In [ ]:
model_sizes = [(1e9, '1B', '#14b8a6'),
               (7e9, '7B', '#6366f1'),
               (7e10, '70B', '#f97316')]

D_grid = np.logspace(9, 13, 200)   # 1B tokens up to 10T tokens

fig, ax = plt.subplots(figsize=(8.5, 5))
for N, label, color in model_sizes:
    C  = 6 * N * D_grid
    L  = chinchilla_loss(N, D_grid, A=A_fit, B=B_fit, E=E_fit)
    ax.loglog(C, L, label=f'{label} params', color=color, linewidth=2)

    # Optimal point: D* = 20N -> C* = 120 N^2.
    D_star = 20 * N
    C_star = 6 * N * D_star
    L_star = chinchilla_loss(N, D_star, A=A_fit, B=B_fit, E=E_fit)
    ax.scatter([C_star], [L_star], marker='*', s=160, color=color,
               edgecolor='#e2e8f0', linewidth=0.8, zorder=5)

ax.set_xlabel('Compute  C = 6 N D  (FLOPs)')
ax.set_ylabel('Loss  L')
ax.set_title('Chinchilla scaling: loss vs compute  (★ = compute-optimal per N)')
ax.legend()
plt.tight_layout(); plt.show()

**What to notice:** on the log-log plot each model size traces its own loss-vs-compute curve, and the
**frontier** (lower envelope) is the compute-optimal path — small models win at small budgets, big
models at big budgets, with the marked optima moving along the envelope. Reading loss off this chart
*before training* is exactly how frontier labs plan runs.

## Gotchas & tradeoffs

- **Chinchilla-optimal ≠ deployment-optimal.** The optimum ignores *inference* cost. If a model will
  serve billions of queries, **over-training a smaller model** (Llama-3: 8B params on 15T tokens ≈
  1,900 tokens/param!) is cheaper overall — the flat optimum makes this nearly free in loss.
- **Power laws are fits, not physics.** Extrapolating far beyond the sweep, or across data
  distributions/architectures, can break; constants differ per setup.
- **Loss ≠ capability.** Downstream abilities can improve unevenly ("emergence" debates); the law
  predicts loss only.
- **`C ≈ 6ND` is itself an approximation** (dense transformer forward+backward); MoE and other
  architectures change the accounting.

In [ ]:
# The optimum is FLAT: deviating 2-4x from N* barely hurts loss but slashes inference cost
C_flat = 1e23                                   # (the plot above reused the name C for an array)
a, b = alpha_true, beta_true
N_star = ((a * A_fit) / (b * B_fit)) ** (1 / (a + b)) * (C_flat / 6) ** (b / (a + b))
L_best = chinchilla_loss(N_star, C_flat / (6 * N_star), A=A_fit, B=B_fit, E=E_fit)
for factor in [0.25, 0.5, 1.0, 2.0, 4.0]:
    N_dev = N_star * factor
    L_dev = chinchilla_loss(N_dev, C_flat / (6 * N_dev), A=A_fit, B=B_fit, E=E_fit)
    print(f'N = {factor:>4}x N*: loss = {L_dev:.4f}  (+{100*(L_dev-L_best)/L_best:.2f}%)   '
          f'inference cost ~ {factor}x')
print('\n-> a 4x-smaller over-trained model gives up ~1-2% loss but is 4x cheaper to serve (the Llama strategy)')

**What to notice:** quartering the model size costs only ~1–2% extra loss at the same training
budget but makes every future inference 4× cheaper — the flatness of the optimum is why post-Chinchilla
practice (Llama, Mistral) deliberately trains *small models on far more* than 20 tokens/param. The law
still governs; the objective just changed from training-compute to lifetime cost.

## Key takeaways

- **Chinchilla loss law:** $L(N, D) = A/N^\alpha + B/D^\beta + E$ — each term
  captures one bottleneck (model capacity, data, irreducible noise).
- **Fitting** $(A, B, E)$ at known exponents is a *linear* least-squares problem
  — one `np.linalg.lstsq` recovers everything.
- **Compute-optimal recipe:** $D^* = 20 N^*$, giving $N^* = \sqrt{C / 120}$,
  $D^* = 20 N^*$ when $C = 6 N D$.
- Plotting $L$ vs $C$ on log-log axes makes power-law scaling visually obvious.

---
## ✏️ Your turn

### Exercise — implement `chinchilla_optimal(C)`

Implement a function that takes a training compute budget $C$ (FLOPs) and
returns the compute-optimal **(N\*, D\*)** under the Chinchilla rule
$D = 20 N$ with $C = 6 N D$.

$$C = 6 N D = 6 N (20 N) = 120 N^2 \;\Rightarrow\; N^* = \sqrt{C / 120}, \quad D^* = 20 N^*.$$

In [ ]:
import math

def chinchilla_optimal(C):
    """
    Return (N_opt, D_opt) for compute budget C (FLOPs) under the Chinchilla
    compute-optimal rule  D = 20 N,  C = 6 N D.

    Args:
        C : float, total training compute in FLOPs.

    Returns:
        (N_opt, D_opt) : floats. Parameter count and training-token count.
    """
    # TODO(you): derive  N* = sqrt(C / 120),  D* = 20 N*.
    ...

In [ ]:
# Test 1: C = 1.2e23 FLOPs  ->  N* should be roughly 3.16e10 (~32B), D* ~ 6.3e11 (~630B).
N_opt, D_opt = chinchilla_optimal(1.2e23)
assert math.isclose(N_opt, math.sqrt(1.2e23 / 120.0), rel_tol=1e-9), \
    f"N_opt mismatch: got {N_opt}"
assert math.isclose(D_opt, 20 * N_opt, rel_tol=1e-12), \
    f"D_opt must equal 20 * N_opt under the Chinchilla rule"

# Test 2: the C = 6 N D identity must hold.
C_check = 6 * N_opt * D_opt
assert math.isclose(C_check, 1.2e23, rel_tol=1e-9), \
    f"6 * N * D must equal C; got {C_check:.3e}"

# Test 3: doubling C scales N* by sqrt(2).
N_lo, _ = chinchilla_optimal(1e22)
N_hi, _ = chinchilla_optimal(2e22)
assert math.isclose(N_hi / N_lo, math.sqrt(2), rel_tol=1e-9), \
    f"Doubling C should multiply N* by sqrt(2); got ratio {N_hi/N_lo:.4f}"

print("✅ chinchilla_optimal is correct")
print(f"   At C = 1.2e23 FLOPs:  N* ≈ {N_opt/1e9:.1f}B params,  D* ≈ {D_opt/1e9:.0f}B tokens")

<details>
<summary>💡 Show solution</summary>

```python
def chinchilla_optimal(C):
    N_opt = math.sqrt(C / 120.0)
    D_opt = 20.0 * N_opt
    return N_opt, D_opt
```

Derivation: substitute $D = 20 N$ into $C = 6 N D$ to get $C = 120 N^2$,
then solve for $N$.

</details>